[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreas-he/ai-safety-challenges/blob/main/challenges/2026-04-20-steering-vectors/challenge.ipynb)

# Steering Vectors — Editing Model Behavior via Residual Stream Directions

**Category:** Mech Interp | **Format:** short-coding | **Difficulty:** standard | **Est. time:** 20–30 min

**Relevance.** Steering vectors (Turner et al. 2023, Zou et al. 2023 "Representation Engineering") are one of the most practically important tools in AI safety: you extract a direction that separates two semantic clusters in activation space, then *add* it at inference time to shift model behavior without any weight update. This is the same mechanism used in refusal-direction ablation experiments and in testing whether sleeper-agent behaviors can be removed via representation-level edits — both core topics for LASR/SAIGE interviews.

**The task.** Implement three pure-numpy utilities that together form a complete steering-vector pipeline:
1. Extract the direction separating two classes of examples (`compute_steering_vector`)
2. Measure how strongly any activation points along that direction (`projection_onto_vector`)
3. Apply the steering intervention at inference time (`steer_activations`)

## Setup

In [ ]:
import numpy as np
np.random.seed(42)

# Synthetic dataset: two semantic clusters in d=16 dimensional activation space.
# Positive examples are shifted +1 along the first axis; negative examples shifted -1.
d = 16
n = 20
signal = np.zeros(d); signal[0] = 1.0

pos_acts = np.random.randn(n, d) + signal    # shape (20, 16)
neg_acts = np.random.randn(n, d) - signal    # shape (20, 16)
test_acts = np.random.randn(n, d)            # unlabeled activations for steering test

## Task 1 — Compute the Steering Vector

The **mean-difference direction** is the simplest and most robust way to extract a semantic direction:

$$v_{\text{raw}} = \text{mean}(A_{\text{pos}}) - \text{mean}(A_{\text{neg}})$$

Normalize to unit L2 norm so downstream dot-products give signed cosine similarities rather than magnitude-dependent numbers.

**Why normalize?** When you add `alpha * v` at inference time, `alpha` should control the *strength* of the intervention, not be entangled with the scale of the direction vector.

In [ ]:
def compute_steering_vector(pos_acts: np.ndarray, neg_acts: np.ndarray) -> np.ndarray:
    """
    Compute the normalized mean-difference steering direction.

    Args:
        pos_acts: shape (n_pos, d) — activations for positive-class examples
        neg_acts: shape (n_neg, d) — activations for negative-class examples
    Returns:
        v: shape (d,) — unit-norm steering vector pointing from neg to pos cluster
    """
    raise NotImplementedError

### Tests — Task 1

In [ ]:
v = compute_steering_vector(pos_acts, neg_acts)

assert v.shape == (d,), f"Expected shape ({d},), got {v.shape}"
assert abs(np.linalg.norm(v) - 1.0) < 1e-6, (
    f"v must be unit-norm, got ||v|| = {np.linalg.norm(v):.6f}"
)
# The true signal direction is axis 0; recovered v should align strongly with it
assert v[0] > 0.8, (
    f"First component should dominate (signal is along axis 0), got v[0] = {v[0]:.4f}"
)

print(f"Task 1 ✓  ||v|| = {np.linalg.norm(v):.6f}, v[0] = {v[0]:.4f}")

## Task 2 — Project Activations onto the Steering Direction

Given a set of activations `acts` (shape `n × d`) and a **unit** steering vector `v` (shape `d`), compute the signed scalar projection of each row onto `v`:

$$p_i = a_i \cdot v$$

Positive values mean the activation points *with* `v` (toward the positive class); negative values mean it points against.

**Why this matters.** Projection scores let you verify that your steering vector actually separates the two clusters before you deploy it as an intervention. They also let you track how much a given layer's activations encode the concept.

In [ ]:
def projection_onto_vector(acts: np.ndarray, v: np.ndarray) -> np.ndarray:
    """
    Project each row of acts onto unit vector v.

    Args:
        acts: shape (n, d)
        v:    shape (d,)  — assumed to be unit norm
    Returns:
        projs: shape (n,) — signed scalar projections
    """
    raise NotImplementedError

### Tests — Task 2

In [ ]:
pos_projs = projection_onto_vector(pos_acts, v)
neg_projs = projection_onto_vector(neg_acts, v)

assert pos_projs.shape == (n,), f"Expected ({n},), got {pos_projs.shape}"
assert pos_projs.mean() > 0, (
    f"Positive examples should project positively, mean = {pos_projs.mean():.4f}"
)
assert neg_projs.mean() < 0, (
    f"Negative examples should project negatively, mean = {neg_projs.mean():.4f}"
)
# The two clusters should be well-separated (signal strength ~2, noise ~1)
assert pos_projs.mean() - neg_projs.mean() > 1.5, (
    f"Expected separation > 1.5, got {pos_projs.mean() - neg_projs.mean():.4f}"
)

print(f"Task 2 ✓  pos mean proj = {pos_projs.mean():.3f}, neg mean proj = {neg_projs.mean():.3f}")

## Task 3 — Apply Steering at Inference Time

The intervention is simple: add `alpha * v` to every row of the activation tensor at a chosen layer:

$$a_i^{\text{steered}} = a_i + \alpha \cdot v$$

`alpha > 0` nudges activations toward the positive class; `alpha < 0` pushes them away.

**Key property.** Because the residual stream is additive, steering only changes the component of each activation along `v` — all orthogonal components are untouched. This is what makes the intervention interpretable: you're not scrambling the representation, you're precisely editing one direction.

In [ ]:
def steer_activations(acts: np.ndarray, v: np.ndarray, alpha: float) -> np.ndarray:
    """
    Add alpha * v to every row of acts.

    Args:
        acts:  shape (n, d)
        v:     shape (d,) — unit steering vector
        alpha: float — intervention strength (positive steers toward pos class)
    Returns:
        steered: shape (n, d)
    """
    raise NotImplementedError

### Tests — Task 3

In [ ]:
alpha = 5.0
steered = steer_activations(neg_acts, v, alpha)

assert steered.shape == neg_acts.shape, (
    f"Shape must be preserved: expected {neg_acts.shape}, got {steered.shape}"
)
# Projections should shift by exactly alpha (since v is unit norm)
orig_projs  = projection_onto_vector(neg_acts, v)
steer_projs = projection_onto_vector(steered, v)
shifts = steer_projs - orig_projs
assert np.allclose(shifts, alpha, atol=1e-6), (
    f"Each projection must shift by alpha={alpha}; got range [{shifts.min():.4f}, {shifts.max():.4f}]"
)
# Orthogonal components must be unchanged
v_perp = np.zeros(d); v_perp[1] = 1.0  # axis 1 is orthogonal to v (v[1] ≈ 0)
orig_perp  = neg_acts @ v_perp
steer_perp = steered @ v_perp
assert np.allclose(orig_perp, steer_perp, atol=1e-6), "Orthogonal components must be unchanged"

print(f"Task 3 ✓  Each projection shifted by exactly alpha={alpha}")
print("All tests passed! 🎉")

---

**Reflection.** You've just implemented the core of *activation addition* — the same operation Zou et al. (2023) use to induce truthfulness, harmlessness, and other abstract concepts in LLMs without touching weights. Notice that the intervention's precision comes entirely from the quality of the direction `v`: if `v` conflates two concepts (e.g. "refusal" and "politeness"), steering will move both. That's why representation engineering papers spend most of their effort on *contrastive dataset curation*, not on the math — the math is three lines.

<details>
<summary><b>Solution</b></summary>

```python
def compute_steering_vector(pos_acts, neg_acts):
    v = pos_acts.mean(axis=0) - neg_acts.mean(axis=0)
    return v / np.linalg.norm(v)

def projection_onto_vector(acts, v):
    return acts @ v  # v is unit norm — dot product = projection

def steer_activations(acts, v, alpha):
    return acts + alpha * v  # broadcasts (n,d) + (d,) correctly
```

**Key insight.** All three functions are one line each once you internalize the shapes. The entire steering-vector pipeline — extract, measure, intervene — reduces to: a mean difference, a matrix-vector product, and an additive shift. The power is not in the arithmetic but in the *choice of contrastive pairs* that define `pos_acts` and `neg_acts`. That dataset curation step determines whether `v` is a clean semantic direction or a noisy mixture of correlated concepts.

In production (Zou et al., Turner et al.): `v` is often averaged across multiple layers, extracted at the token position of interest (e.g. the last token of the harmful prompt), and scaled to match the activation norm at the target layer. The numpy primitives here are identical to what runs inside those experiments.

</details>